# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Agent",
    instructions= """
    You are  Nutrition agent assistant
    You will provide concise results
    """

)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Agent", ...)
- Final output (str):
    Short answer: generally very healthy and convenient.
    
    Key points per medium banana (about 118 g):
    - Calories: ~105
    - Carbs: ~27 g (1–2 g fiber)
    - Potassium: ~420 mg (about 14% DV)
    - Vitamin B6: ~20% DV
    - Vitamin C: ~15% DV
    - Low in fat; contains natural sugars
    
    Benefits:
    - Supports blood pressure and heart health (potassium)
    - Quick energy (carbs) and dietary fiber for digestion
    - Provides vitamin B6 and vitamin C
    
    Considerations:
    - Higher in sugar than some fruits; portion control if managing blood sugar
    - Unripe bananas have more resistant starch (lower GI); ripe have sweeter taste
    
    Bottom line: as part of a balanced diet, bananas are a healthy, convenient option. 1–2 per day fits many diets.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Short answer: generally very healthy as part of a balanced diet.

Key nutrients (per medium banana, ~118 g):
- Potassium: ~420 mg (supports heart and fluid balance)
- Vitamin B6: ~0.4 mg
- Vitamin C: ~10 mg
- Dietary fiber: ~3 g
- Carbohydrates: mainly natural sugars and starch (supports quick energy)

Benefits:
- May improve digestion and satiety
- Supports heart health and blood pressure
- Moderates blood sugar when eaten with other foods (glycemic index is moderate)

Considerations:
- About 100–110 calories per fruit; sugar content varies by ripeness.
- Unripe to ripe: resistant starch declines and sugars increase.
- Kidney disease patients may need potassium limits; check with a clinician.
- Allergies are rare but possible.

Bottom line: a healthy, convenient fruit rich in potassium and fiber; fit for most diets. If you have specific conditions (diabetes, kidney disease), adjust portions accordingly.

_Good Job!_